In [ ]:
# notebook to find multiTF enhancer instances where gtex eQTLs overlap multiTF enhancers

In [1]:
# import packages
import pandas as pd
import os
from tqdm import tqdm
import numpy as np
import gc
import psutil
import pickle

In [2]:
### preprocessing ###
# # iterate through all openTargets data
# openTargets2cat = []
# for i in tqdm([i for i in os.listdir(path2openTargets) if i.endswith('.parquet')]):
#     parqLife = pd.read_parquet(f'{path2openTargets}/{i}')                                                                                                                                                                  
#     openTargets2cat.append(parqLife)
# openTargetsCredibleSets = pd.concat(openTargets2cat)

# # clean up
# del openTargets2cat
# gc.collect()
# # merge study IDs with credible set data
# openTargetsMerged = openTargetsCredibleSets.merge(
#     openTargetsStudyInfo[['studyId', 'traitFromSource', 'traitFromSourceMappedIds', 'pubmedId', 'initialSampleSize',
#                           'cohorts','nCases', 'nControls', 'nSamples', 'diseaseIds']], 
#     on='studyId', 
#     how='left'
# )

# # clean up
# del openTargetsCredibleSets, openTargetsStudyInfo
# gc.collect()
# # filter for only gwas
# openTargetsGWAS = openTargetsMerged[openTargetsMerged['studyType'] == 'gwas'].copy()
# # rename the variantID to leadVariant - will have a duplicate columname
# openTargetsGWAS = openTargetsGWAS.rename(columns={'variantId' : 'leadVariant'})

# # clean up
# del openTargetsMerged
# gc.collect()

In [3]:
### all GWAS credible sets were opened and exploded chunkwise with openTargets_explode.py
### then those were filtered for high (> 0.9) PIP and mid (> 0.5) PIP variants
### here we will open those filtered variants and intersect them with the multiTF enhancers/emVars

In [4]:
# open summary multiTF data
all_dELS_emVars = pd.read_csv('results_final/multiTF_variants.tsv', sep = '\t')
# filter for only those in multiTF enhancers
multiTF_emVars = all_dELS_emVars[all_dELS_emVars['is_multiTF'] == True].copy()
# add gtex id
multiTF_emVars.loc[:,'openTargets_id'] = [('_').join([chrom.split('chr')[-1], str(pos), ref, alt]) for chrom, pos, ref, alt in zip(multiTF_emVars['chrom'],
                                                                                                                                   multiTF_emVars['pos'],
                                                                                                                                   multiTF_emVars['ref'],
                                                                                                                                   multiTF_emVars['alt'])]

In [5]:
# load padding-zone emVars (4bp flanking filtered seqlets, not captured by exact BED intersection)
padding_emVars_path = 'results_final/padding_emVars_gnomAD_v4.tsv'
if os.path.exists(padding_emVars_path):
    padding_emVars = pd.read_csv(padding_emVars_path, sep='\t')
    padding_cols = ['variant_id', 'chrom', 'pos', 'ref', 'alt', 'cell_type', 'skew_pred', 'enhancer_ids', 'is_multiTF']
    multiTF_emVars = pd.concat(
        [multiTF_emVars, padding_emVars[padding_cols]], ignore_index=True
    ).drop_duplicates(subset=['variant_id', 'cell_type'], keep='first')
    # recompute openTargets_id for new variants
    multiTF_emVars.loc[:,'openTargets_id'] = [
        ('_').join([chrom.split('chr')[-1], str(pos), ref, alt])
        for chrom, pos, ref, alt in zip(
            multiTF_emVars['chrom'], multiTF_emVars['pos'],
            multiTF_emVars['ref'], multiTF_emVars['alt']
        )
    ]
    print(f'Added {len(padding_emVars)} padding emVars → {len(multiTF_emVars)} total multiTF emVars')
else:
    print(f'Padding emVars file not found: {padding_emVars_path} — skipping')

Padding emVars file not found: results_final/padding_emVars_gnomAD_v4.tsv — skipping


In [6]:
# open the filtered opentargets data
# high PIP
# store the path to the open targets data as a variable (credible_sets)
path2highPipOpenTargets = 'exploded_chunks/highPIP_09/'
# mid PIP
path2midPipOpenTargets = 'exploded_chunks/midPIP_05/'

In [7]:
# open all high pip variants from open targets (>0.9)
highPipOpenTargets2cat = []
for i in tqdm([i for i in os.listdir(path2highPipOpenTargets) if i.endswith('.parquet')]):
    parqLife = pd.read_parquet(f'{path2highPipOpenTargets}/{i}')
    highPipOpenTargets2cat.append(parqLife)                                                                                                                                                          
# make df
highPipOpenTargetsAll = pd.concat(highPipOpenTargets2cat)

  2%|▏         | 1/58 [00:00<00:08,  6.74it/s]

100%|██████████| 58/58 [00:06<00:00,  8.32it/s]


In [8]:
# open all mid pip variants from open targets (>0.5)
midPipOpenTargets2cat = []
for i in tqdm([i for i in os.listdir(path2midPipOpenTargets) if i.endswith('.parquet')]):
    parqLife = pd.read_parquet(f'{path2midPipOpenTargets}/{i}')
    midPipOpenTargets2cat.append(parqLife)                                                                                                                                                          
# make df
midPipOpenTargetsAll = pd.concat(midPipOpenTargets2cat)

  9%|▊         | 5/58 [00:01<00:10,  4.89it/s]

100%|██████████| 58/58 [00:11<00:00,  5.24it/s]


In [9]:
# filter all high PIP variants for those that are in multiTF emVars
# high PIP
highPipOpenTargetsEmVars = highPipOpenTargetsAll[highPipOpenTargetsAll['leadVariant'].isin(multiTF_emVars['openTargets_id'])]
# mid PIP
midPipOpenTargetsEmVars = midPipOpenTargetsAll[midPipOpenTargetsAll['leadVariant'].isin(multiTF_emVars['openTargets_id'])]

In [10]:
# merge multiTFs into a single DF
# high PIP
highPipOpenTargetsMerge = highPipOpenTargetsEmVars.merge(multiTF_emVars.filter(['variant_id', 'cell_type', 'skew_pred', 'enhancer_ids', 'openTargets_id']), left_on='leadVariant', right_on='openTargets_id', how='inner')
# mid PIP
midPipOpenTargetsMerge = midPipOpenTargetsEmVars.merge(multiTF_emVars.filter(['variant_id', 'cell_type', 'skew_pred', 'enhancer_ids', 'openTargets_id']), left_on='leadVariant', right_on='openTargets_id', how='inner')

In [11]:
# openTargetsAllele2cat = []
# # iterate through chromosomes of high pip targets and add allele frequency
# for chrom in tqdm(highPipOpenTargetsMerge['chromosome'].unique()):
#     # open the filtered gnomAD data
#     print(f'loading chr{chrom} gnomAD data')
#     chromAD = pd.read_csv(f'/projects/tewhey-lab/buttsj/Variant_Effects/gnomad/gnomad_common_vars/filtered_variants/chr{chrom}_gnomad_pass_variants.csv.gz')
#     print(f'loaded chr{chrom} gnomAD data')
#     # change the id for matching the variants in open targets
#     chromAD.loc[:, 'openTargets_id'] = [i.split('chr')[-1] for i in chromAD['ID']]
#     # make dictionaries for annotating open targets data
#     afDict = dict(zip(
#         chromAD['openTargets_id'],
#         chromAD['AF']
#     ))
#     catDict = dict(zip(
#         chromAD['openTargets_id'],
#         chromAD['category']
#     ))
#     # filter the highPIP targets
#     chromOpenTargets = highPipOpenTargetsMerge[highPipOpenTargetsMerge['chromosome'] == chrom].copy()
#     # add the allele frequency
#     chromOpenTargets.loc[:, 'af'] = [afDict.get(i) for i in chromOpenTargets['leadVariant']]
#     # add the allele frequency category
#     chromOpenTargets.loc[:, 'af_category'] = [catDict.get(i) for i in chromOpenTargets['leadVariant']]
#     openTargetsAllele2cat.append(chromOpenTargets)
# # concatenate into single DF
# highPipOpenTargetsMerge_gnomAD_af = pd.concat(openTargetsAllele2cat)
### lots of missing data with this appproach - stick with the allele frequencies from open targets, appears to be from gnomad v4

In [12]:
# open the allele frequency data
openTargets_allele_frequency = pd.read_csv('openTargets_allele_frequencies.tsv.gz', sep = '\t')
# filter for those variants that are in the high PIP subset
highPIP_allele_frequencies = openTargets_allele_frequency[openTargets_allele_frequency['variantId'].isin(highPipOpenTargetsMerge['leadVariant'].tolist())]

In [13]:
# add allele frequency data high PIP variants
highPipOpenTargetsMerge_AF = highPipOpenTargetsMerge.merge(highPIP_allele_frequencies, left_on='openTargets_id', right_on='variantId', how='inner')
# add max allele frequency
highPipOpenTargetsMerge_AF.loc[:, 'AF_Max'] = [np.max([afr, ami, amr, asj, eas, fin, mid, nfe, remain, sas]) for afr, ami, amr, asj, eas, fin, mid, nfe, remain, sas in zip(highPipOpenTargetsMerge_AF['AF_afr_adj'],
                                                                                                                                                                        highPipOpenTargetsMerge_AF['AF_ami_adj'],
                                                                                                                                                                        highPipOpenTargetsMerge_AF['AF_amr_adj'],
                                                                                                                                                                        highPipOpenTargetsMerge_AF['AF_asj_adj'],
                                                                                                                                                                        highPipOpenTargetsMerge_AF['AF_eas_adj'],
                                                                                                                                                                        highPipOpenTargetsMerge_AF['AF_fin_adj'],
                                                                                                                                                                        highPipOpenTargetsMerge_AF['AF_mid_adj'],
                                                                                                                                                                        highPipOpenTargetsMerge_AF['AF_nfe_adj'],
                                                                                                                                                                        highPipOpenTargetsMerge_AF['AF_remaining_adj'],
                                                                                                                                                                        highPipOpenTargetsMerge_AF['AF_sas_adj'])]

In [14]:
# save to disk
# highPipOpenTargetsMerge_AF.to_csv('OpenTargets_MultiTF_HighPip_emVars.tsv', sep = '\t', index = False)

In [15]:
### phase 2 ###
# get all other emVars in the enhancers in the other TF/in the same TF to find instances where allele frequencies differ.
# for now let's annotate with gnomAD v3 (from Stephen's analysis) - check into v4 shortly
# open the pickle
with open('results_final/multiTF_analysis.pkl', 'rb') as f:
    multiTF_pickle = pickle.load(f)
# open the multiTF summary file
multiTF_summary = pd.read_csv('results_final/multiTF_summary.tsv', sep = '\t')
# make a dictionary of enhancer ID : lead variant pairs
enhID_leadVarDict = dict(zip(highPipOpenTargetsMerge_AF['enhancer_ids'], highPipOpenTargetsMerge_AF['leadVariant']))
# go back the other way - leadVariant to enhancer
leadVar_enhID = dict(zip(highPipOpenTargetsMerge_AF['leadVariant'], highPipOpenTargetsMerge_AF['enhancer_ids']))

In [16]:
  # Pre-group for O(1) lookups instead of filtering the full df each time                                                                                                                                                                                                                                             
  emvar_groups = multiTF_emVars.groupby(['cell_type', 'enhancer_ids'])                                                                                                                                                                                                                                                
                                                                                                                                                                                                                                                                                                                      
  allSeqletEmvars2cat = []                                                                                                                                                                                                                                                                                            
                                                                                                                                                                                                                                                                                                                      
  for leadVar in tqdm(highPipOpenTargetsMerge_AF['leadVariant'].unique()):                                                                                                                                                                                                                                            
      enhancer = leadVar_enhID.get(leadVar)                                                                                                                                                                                                                                                                           
      enhancerCells = list(multiTF_summary[multiTF_summary['enhancer_id'] == enhancer]['cell_type'].unique())                                                                                                                                                                                                         
                                                                                                                                                                                                                                                                                                                      
      for cell in enhancerCells:                                                                                                                                                                                                                                                                                      
          filteredSeqlets = multiTF_pickle['multiTF_enhancers'][cell][enhancer]['filtered_seqlets']                                                                                                                                                                                                                   
                                                                                                                                                                                                                                                                                                                      
          # Get all variants for this cell/enhancer combo once                                                                                                                                                                                                                                                        
          try:                                                                                                                                                                                                                                                                                                        
              cell_enh_vars = emvar_groups.get_group((cell, enhancer))                                                                                                                                                                                                                                                
          except KeyError:                                                                                                                                                                                                                                                                                            
              continue                                                                                                                                                                                                                                                                                                
                                                                                                                                                                                                                                                                                                                      
          cell_seqlet_vars = []                                                                                                                                                                                                                                                                                       
          tfCount = 1                                                                                                                                                                                                                                                                                                 
          lead_found = False                                                                                                                                                                                                                                                                                          
                                                                                                                                                                                                                                                                                                                      
          for start, end, repTF, contrib in zip(filteredSeqlets['start'], filteredSeqlets['end'],                                                                                                                                                                                                                     
                                                 filteredSeqlets['vierstra_cluster'], filteredSeqlets['rep_tf_contrib']):                                                                                                                                                                                            
              # Now just filter by position (already filtered by cell/enhancer)                                                                                                                                                                                                                                       
              seqletVars = cell_enh_vars[(cell_enh_vars['pos'] >= start - 4) & (cell_enh_vars['pos'] <= end + 4)].copy()                                                                                                                                                                                                      
                                                                                                                                                                                                                                                                                                                      
              if len(seqletVars) == 0:                                                                                                                                                                                                                                                                                
                  tfCount += 1                                                                                                                                                                                                                                                                                        
                  continue                                                                                                                                                                                                                                                                                            
                                                                                                                                                                                                                                                                                                                      
              # Scalar assignment is faster than list comprehension                                                                                                                                                                                                                                                   
              seqletVars['tf_family'] = repTF                                                                                                                                                                                                                                                                         
              seqletVars['tf_contrib'] = contrib                                                                                                                                                                                                                                                                      
              seqletVars['tf_instance'] = tfCount                                                                                                                                                                                                                                                                     
              seqletVars['leadVariant'] = leadVar                                                                                                                                                                                                                                                                     
              seqletVars['is_leadVariant'] = (seqletVars['openTargets_id'] == leadVar).astype(int)                                                                                                                                                                                                                    
                                                                                                                                                                                                                                                                                                                      
              if seqletVars['is_leadVariant'].sum() > 0:                                                                                                                                                                                                                                                              
                  lead_found = True                                                                                                                                                                                                                                                                                   
                                                                                                                                                                                                                                                                                                                      
              cell_seqlet_vars.append(seqletVars)                                                                                                                                                                                                                                                                     
              tfCount += 1                                                                                                                                                                                                                                                                                            
                                                                                                                                                                                                                                                                                                                      
          # Only concat and append if lead variant was found in this cell's seqlets                                                                                                                                                                                                                                   
          if lead_found and cell_seqlet_vars:                                                                                                                                                                                                                                                                         
              allSeqletEmvars2cat.append(pd.concat(cell_seqlet_vars).sort_values(by='pos'))                                                                                                                                                                                                                           
                                                                                                                                                                                                                                                                                                                      
  allSeqletEmVars = pd.concat(allSeqletEmvars2cat, ignore_index=True)

  0%|          | 0/210 [00:00<?, ?it/s]

100%|██████████| 210/210 [00:11<00:00, 18.84it/s]


In [17]:
from io import StringIO                                                                                                                                                 
import subprocess
                                                                                                                                                                             
af_pop_cols = ['AF', 'AF_afr', 'AF_ami', 'AF_amr', 'AF_asj', 'AF_eas',
                 'AF_fin', 'AF_mid', 'AF_nfe', 'AF_remaining', 'AF_sas']

af_matches = []

for chrom in tqdm(allSeqletEmVars['chrom'].unique()):
    chrom_vars = allSeqletEmVars[allSeqletEmVars['chrom'] == chrom]
    positions = chrom_vars['pos'].unique()

    # Build grep pattern for positions
    pattern = '|'.join([f'^{chrom}\t{pos}\t' for pos in positions])

    filepath = f'/projects/tewhey-lab/buttsj/Variant_Effects/gnomad/gnomad_v4/filtered_data/popLevel/{chrom}_gnomAD_v4_pass_popLevel_af.tsv.gz'

    # Get header first
    header_cmd = f"zcat {filepath} | head -1"
    header = subprocess.run(header_cmd, shell=True, capture_output=True, text=True).stdout.strip().split('\t')

    # Use zgrep to extract matching lines
    grep_cmd = f"zcat {filepath} | grep -E '{pattern}'"
    result = subprocess.run(grep_cmd, shell=True, capture_output=True, text=True)

    if result.stdout:
        chunk_df = pd.read_csv(StringIO(result.stdout), sep='\t', names=header, na_values='.')
        # openTargets format: no 'chr' prefix, no '_b38' suffix
        chunk_df['id'] = (chunk_df['CHROM'].str.replace('chr', '') + '_' +
                           chunk_df['POS'].astype(str) + '_' +
                           chunk_df['REF'] + '_' + chunk_df['ALT'])
        af_matches.append(chunk_df[['id'] + af_pop_cols])

all_af_data = pd.concat(af_matches, ignore_index=True)

alleSeqletEmVars_AF = allSeqletEmVars.merge(
      all_af_data, left_on='openTargets_id', right_on='id', how='left'
  ).drop(columns=['id']).rename(columns={'AF': 'af'})

  0%|          | 0/22 [00:00<?, ?it/s]

100%|██████████| 22/22 [17:20<00:00, 47.32s/it]


In [18]:
len(alleSeqletEmVars_AF)

5666

In [19]:
  # Select the columns you need from highPipOpenTargetsMerge                                                                                                                                                              
  phenotype_info = highPipOpenTargetsMerge[['leadVariant', 'studyId', 'traitFromSource', 'posteriorProbability', 'is95CredibleSet', 'finemappingMethod']].drop_duplicates()                                                                                                               
                                                                                                                                                                                                                          
  # Check how many study associations exist                                                                                                                                                                               
  print(f"Unique leadVariants: {phenotype_info['leadVariant'].nunique()}")                                                                                                                                                
  print(f"Total variant-study pairs: {len(phenotype_info)}")                                                                                                                                                              
                                                                                                                                                                                                                          
  # Merge onto your allele frequency data                                                                                                                                                                                 
  alleleSeqletEmVars_AF_pheno = alleSeqletEmVars_AF.merge(                                                                                                                                                                    
      phenotype_info,                                                                                                                                                                                                     
      on='leadVariant',                                                                                                                                                                                                   
      how='left'                                                                                                                                                                                                          
  ).sort_values(by=['cell_type', 'enhancer_ids', 'pos'])                                                                                                                                                                                                                       
                                                                                                                                                                                                                          
  print(f"Rows before merge: {len(alleSeqletEmVars_AF)}")                                                                                                                                                                  
  print(f"Rows after merge: {len(alleleSeqletEmVars_AF_pheno)}")

Unique leadVariants: 210
Total variant-study pairs: 1136
Rows before merge: 5666
Rows after merge: 48952


In [20]:
# # save this iteration to disk
# alleleSeqletEmVars_AF_pheno.to_csv('openTargets_GWAS_multiTF_emVars_highPIP_gnomAD_v4_annotated.tsv', sep = '\t', index = False)

In [21]:
# annotate all emVars with lead variant skew comparison (no phenocopy filter - keep all rows)
# get absolute effect size column
alleleSeqletEmVars_AF_pheno['abs_skew'] = alleleSeqletEmVars_AF_pheno['skew_pred'].abs()
# group by context (NOT by tf_instance — lead variant may only be in one seqlet)
group_cols = ['enhancer_ids', 'cell_type', 'traitFromSource', 'studyId']
# get lead variant effect per group
lead_effects = (alleleSeqletEmVars_AF_pheno[alleleSeqletEmVars_AF_pheno['is_leadVariant'] == 1]
                .groupby(group_cols)['abs_skew']
                .first()
                .rename('lead_abs_skew'))
# get max effect per group
max_effects = (alleleSeqletEmVars_AF_pheno
               .groupby(group_cols)['abs_skew']
               .max()
               .rename('max_abs_skew'))
# combine lead and max effects
effect_comparison = pd.concat([lead_effects, max_effects], axis=1).dropna()
effect_comparison['skew_ratio'] = effect_comparison['max_abs_skew'] / effect_comparison['lead_abs_skew']

# merge annotations onto ALL variants (no filtering by max > lead)
all_emVars_annotated = alleleSeqletEmVars_AF_pheno.merge(
    effect_comparison.reset_index(),
    on=group_cols,
    how='inner'
)
# add column flagging if this variant's abs_skew exceeds the lead variant's abs_skew
all_emVars_annotated['exceeds_lead_skew'] = all_emVars_annotated['abs_skew'] > all_emVars_annotated['lead_abs_skew']

print(f'Total annotated rows: {len(all_emVars_annotated)}')
print(f'Contexts where lead variant is not strongest: {effect_comparison[effect_comparison["max_abs_skew"] > effect_comparison["lead_abs_skew"]].shape[0]}')
print(f'Variants exceeding lead skew: {all_emVars_annotated["exceeds_lead_skew"].sum()}')

Total annotated rows: 48952
Contexts where lead variant is not strongest: 1020
Variants exceeding lead skew: 17134


In [22]:
all_emVars_annotated.to_csv('openTargets_GWAS_multiTF_allEmVars_highPIP_gnomAD_v4_annotated_4bp_pad.tsv', sep = '\t', index = False)

In [ ]:
### SCRATCH BELOW ###

In [ ]:
multiTF_summary[multiTF_summary['enhancer_id'] == 'EH38E4014272']

In [ ]:
multiTF_pickle['multiTF_enhancers']['k562']['EH38E4014272']['filtered_seqlets']

In [ ]:
highPipOpenTargetsMerge[highPipOpenTargetsMerge['traitFromSource'] == 'Red blood cell (erythrocyte) distribution width (UKB data field 30070)']

In [ ]:
highPipOpenTargetsMerge.iloc[152, :]

In [ ]:
multiTF_summary[multiTF_summary['enhancer_id'] == 'EH38E2864258']

In [ ]:
k562_EH38E2864258_vars = multiTF_pickle['variants_all'][(multiTF_pickle['variants_all']['enhancer_ids'] == 'EH38E2864258') & (multiTF_pickle['variants_all']['cell_type'] == 'k562')]

In [ ]:
k562_EH38E2864258_vars[(k562_EH38E2864258_vars['pos'] >= 211630413) & (k562_EH38E2864258_vars['pos'] <= 211630420)]